In [1]:
pip install torch torchvision pycocotools matplotlib


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
from PIL import Image
from pycocotools.coco import COCO
from tqdm import tqdm
import time

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as F
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torch.cuda.amp import autocast, GradScaler

# ------------------ Dataset for COCO format ------------------

class CocoDataset(Dataset):
    def __init__(self, image_dir, annotation_file, transforms=None):
        self.image_dir = image_dir
        self.coco = COCO(annotation_file)
        self.ids = list(self.coco.imgs.keys())
        self.transforms = transforms

    def __getitem__(self, idx):
        image_info = self.coco.loadImgs(self.ids[idx])[0]
        image_path = os.path.join(self.image_dir, image_info['file_name'])
        image = Image.open(image_path).convert("RGB")

        ann_ids = self.coco.getAnnIds(imgIds=image_info['id'])
        annots = self.coco.loadAnns(ann_ids)

        boxes, labels = [], []
        for obj in annots:
            if 'bbox' in obj and obj['bbox']:
                x, y, width, height = obj['bbox']
                if width > 0 and height > 0:
                    boxes.append([x, y, x + width, y + height])
                    labels.append(obj['category_id'])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        if boxes.ndim != 2 or boxes.shape[0] == 0:
            raise IndexError(f"No valid boxes for index {idx}")

        image_id = torch.tensor([image_info['id']])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd,
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

    def __len__(self):
        return len(self.ids)

def collate_fn(batch):
    return tuple(zip(*batch))

# ------------------ Save and Load ------------------

def save_preprocessed_data(dataset, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    for i in tqdm(range(len(dataset)), desc="📦 Saving preprocessed data"):
        try:
            image, target = dataset[i]
            torch.save((image, target), os.path.join(save_dir, f"data_{i}.pt"))
        except IndexError as e:
            print(f"⚠️ Skipping index {i}: {e}")

class LazyCustomDataset(Dataset):
    def __init__(self, save_dir):
        self.save_dir = save_dir
        self.files = sorted([f for f in os.listdir(save_dir) if f.endswith(".pt")])
        if not self.files:
            raise ValueError(f"❌ No .pt files found in {save_dir}. Run preprocessing first.")
        print(f"✅ Found {len(self.files)} preprocessed samples.")

    def __getitem__(self, idx):
        return torch.load(os.path.join(self.save_dir, self.files[idx]))

    def __len__(self):
        return len(self.files)

# ------------------ Model ------------------

def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# ------------------ Paths ------------------
image_dir = r"E:/IMAGE/SWITCH_RACK/Switch_Rack.v1i.coco/train"
annotation_file = r"E:/IMAGE/SWITCH_RACK/Switch_Rack.v1i.coco/train_coco/_annotations.coco.json"
save_dir = r"E:/IMAGE/SWITCH_RACK/processed_data"

# ------------------ Preprocessing if needed ------------------

if not os.path.exists(save_dir) or len(os.listdir(save_dir)) == 0:
    print("⚙️ No preprocessed data found. Starting preprocessing...")
    coco_dataset = CocoDataset(image_dir, annotation_file, transforms=F.to_tensor)
    save_preprocessed_data(coco_dataset, save_dir)
else:
    print("✅ Preprocessed data already exists.\n")

# ------------------ Lazy Loading Dataset ------------------

print("📂 Initializing lazy dataset and dataloader ...")
dataset = LazyCustomDataset(save_dir)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
print(f"✅ DataLoader ready with {len(dataset)} samples.\n")

# ------------------ Model & Training ------------------

num_classes = 3  # 2 classes + background
model = get_model(num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10

# Initialize GradScaler for mixed precision if CUDA is available
if torch.cuda.is_available():
    scaler = GradScaler()
    mixed_precision = True
else:
    mixed_precision = False

# ------------------ Training Loop ------------------

print("🚀 Starting training ...")
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    epoch_start_time = time.time()

    for batch_idx, (images, targets) in enumerate(dataloader):
        batch_start_time = time.time()

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        if mixed_precision:
            with autocast():
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            losses.backward()
            optimizer.step()

        epoch_loss += losses.item()

        batch_end_time = time.time()
        batch_duration = batch_end_time - batch_start_time
        print(f"[Epoch {epoch+1}/{num_epochs}] 🧾 Batch {batch_idx+1}/{len(dataloader)} | Loss: {losses.item():.4f} | Batch Time: {batch_duration:.2f}s")

    epoch_end_time = time.time()
    epoch_duration = epoch_end_time - epoch_start_time
    print(f"\n✅ Epoch {epoch+1} completed. Total Loss: {epoch_loss:.4f} | Epoch Time: {epoch_duration:.2f}s")

    model_path = f"fasterrcnn_switch_model_epoch{epoch+1}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"💾 Model saved to {model_path}\n")

print("🎉 Training complete!")


✅ Preprocessed data already exists.

📂 Initializing lazy dataset and dataloader ...
✅ Found 4923 preprocessed samples.
✅ DataLoader ready with 4923 samples.

🚀 Starting training ...
[Epoch 1/10] 🧾 Batch 1/308 | Loss: 2.0568 | Batch Time: 659.08s
[Epoch 1/10] 🧾 Batch 2/308 | Loss: 0.4265 | Batch Time: 268.62s
[Epoch 1/10] 🧾 Batch 3/308 | Loss: 0.2069 | Batch Time: 241.12s
[Epoch 1/10] 🧾 Batch 4/308 | Loss: 0.2337 | Batch Time: 295.40s
[Epoch 1/10] 🧾 Batch 5/308 | Loss: 0.2243 | Batch Time: 299.29s
[Epoch 1/10] 🧾 Batch 6/308 | Loss: 0.2305 | Batch Time: 277.36s
[Epoch 1/10] 🧾 Batch 7/308 | Loss: 0.2239 | Batch Time: 236.73s
